# 01. The anchor

The regulator publishes how prices moved; the census publishes what owners say their homes are worth.
Every dollar in the story is the census median value in one anchor year moved by the index. This
notebook shows why the anchor year is a decision and keeps the dead end: the first version anchored
every metro in the latest release, which made the self report bias of one survey year the level of
every year before it, and moved every headline when the anchor was moved.

First on the simulator, where the true value and the planted self report bias are known; then the real
metros from the committed marts and the manifest.

In [1]:
import json
import os
from pathlib import Path

import duckdb
import numpy as np
import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_fmt_str_lengths(90)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_float_precision(3)
ROOT = Path(os.environ.get("DOORSTEP_NOTEBOOK_ROOT") or (Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent))
manifest = json.loads((ROOT / "results" / "manifest.json").read_text(encoding="utf-8"))
values, tables = manifest["values"], manifest["tables"]


def v(key):
    return values[key]["value"]


def t(key):
    # A manifest value formatted the way the documents print it.
    from doorstep_core.formats import format_value

    return format_value(values[key]["value"], values[key]["fmt"])


def table(key):
    entry = tables[key]
    return pl.DataFrame(entry["rows"], schema=entry["columns"], orient="row")


MARTS = ROOT / "results" / "marts"
print("manifest as of", manifest["as_of"], "with", len(values), "values; prices to", v("data.fhfa.latest_period"), "and the census to", v("data.acs.last_year"))

manifest as of 2026-10-22 with 923 values; prices to 2026Q2 and the census to 2024


## On the simulator

Each simulated metro reports its value with a known factor over the true one. Anchoring in any one
year carries that factor into every year, so the anchored series is the true series times the metro's
bias; the census value over the transaction values in the loan extract bounds the factor without
removing it.

In [2]:
from doorstep_sim.metros import DEMONSTRATION, generate

world = generate(DEMONSTRATION, 7)
my = world.metro_year
truth = world.truth.metros.select("metro_id", "value_bias")
reported = (
    my.filter(pl.col("year").is_in([2015, 2019, 2023]))
    .select("metro_id", "year", "median_value_reported", "median_value_true")
    .with_columns((pl.col("median_value_reported") / pl.col("median_value_true")).alias("reported_over_true"))
    .join(truth, on="metro_id")
)
reported.group_by("year").agg(
    pl.col("reported_over_true").mean().alias("mean reported over true"),
    (pl.col("reported_over_true") - pl.col("value_bias")).abs().mean().alias("mean gap to the planted factor"),
).sort("year")

year,mean reported over true,mean gap to the planted factor
i64,f64,f64
2015,1.004,0.012
2019,1.005,0.013
2023,1.003,0.011


The gap to the planted factor is the survey's own sampling noise; the factor itself is what the
anchor inherits. The chapter's self report correction estimates it from the loan extract and the
recovery study grades it.

## The dead end: anchoring in the latest year

The first version anchored in the latest release. Every headline then rested on one survey year that
had not finished revising, and when the anchor was moved to 2019 the national payment share moved
with it. The sensitivity table in the manifest prints the headline under every anchor; the policy
anchor is 2019, the last full response year before the pandemic.

In [3]:
table("ch1.anchor_sensitivity")

Anchor year,Latest payment share,2006 payment share,Difference,Metros above 2006,Note
i64,f64,f64,f64,i64,str
2015,0.269,0.243,0.026,47,""""""
2017,0.267,0.241,0.026,47,""""""
2019,0.268,0.242,0.026,47,"""chosen"""
2023,0.266,0.240,0.026,47,""""""


## The real metros

The national series from the chapter, and the same series re-anchored in each year of the table above,
computed here from the marts so the arithmetic is visible: value in the anchor year times the index
ratio, the payment at the quarterly rate on ninety percent of it, twelve payments over the median
income, weighted by owner households.

In [4]:
from doorstep_finance.mortgage import payment_share

metro_year = pl.read_parquet(MARTS / "mart_metro_year.parquet").filter(pl.col("kind") == "metropolitan")


def national_payment_share(frame, anchor_year):
    base = frame.filter((pl.col("year") == anchor_year) & pl.col("median_value_reported").is_not_null() & pl.col("index_at").is_not_null()).select(
        "cbsa_code", pl.col("median_value_reported").alias("anchor_value"), pl.col("index_at").alias("anchor_index")
    )
    rows = (
        frame.join(base, on="cbsa_code")
        .filter(pl.col("index_at").is_not_null() & (pl.col("median_income") > 0) & pl.col("rate").is_not_null() & pl.col("owner_households").is_not_null())
        .with_columns((pl.col("anchor_value") * pl.col("index_at") / pl.col("anchor_index")).alias("value"))
    )
    out = []
    for year, group in sorted(rows.group_by("year"), key=lambda kv: kv[0][0]):
        w = group["owner_households"].to_numpy()
        shares = np.array([payment_share(v, i, r, 0.10, 30) for v, i, r in zip(group["value"], group["median_income"], group["rate"])])
        out.append({"year": year[0], "anchor": anchor_year, "payment_share": float(np.average(shares, weights=w)), "metros": group.height})
    return pl.DataFrame(out)


series = pl.concat([national_payment_share(metro_year, a) for a in (2015, 2017, 2019, 2023)])
series.pivot(on="anchor", index="year", values="payment_share").sort("year")

year,2015,2017,2019,2023
i64,f64,f64,f64,f64
2005,0.275,0.260,0.241,0.191
2006,0.276,0.260,0.242,0.191
2007,0.276,0.260,0.242,0.191
2008,0.265,0.250,0.232,0.183
2009,0.217,0.204,0.190,0.150
2010,0.200,0.188,0.175,0.138
2011,0.191,0.179,0.166,0.131
2012,0.185,0.174,0.162,0.127
2013,0.187,0.176,0.163,0.129


In [5]:
latest = series.filter(pl.col("year") == series["year"].max())
peak = series.filter(pl.col("year") == 2006)
latest.join(peak, on="anchor", suffix="_2006").select(
    "anchor", pl.col("payment_share").alias("latest"), pl.col("payment_share_2006").alias("in 2006"), (pl.col("payment_share") - pl.col("payment_share_2006")).alias("difference")
)

anchor,latest,in 2006,difference
i64,f64,f64,f64
2015,0.309,0.276,0.032
2017,0.290,0.260,0.029
2019,0.268,0.242,0.026
2023,0.210,0.191,0.019


The difference against 2006 is the headline of chapter 1. It is read on the anchor the policy fixed
before the data; the table shows what each other anchor would have said, and the chapter prints the
word stable or degenerate depending on whether the sign survives the first neighbour.